# Social Network Ads — EDA y Modelo de Regresión Logística

**Dataset:** `social_network_ads (2).csv` — 400 usuarios de una red social a los que se les
mostró un anuncio de un producto. Se registra si terminaron comprándolo.

| Variable | Tipo | Rol |
|---|---|---|
| `User ID` | identificador | se descarta |
| `Gender` | categórica **dicotómica** | predictora |
| `Age` | numérica continua | predictora |
| `EstimatedSalary` | numérica continua | predictora |
| `Purchased` | **dicotómica** (0/1) | variable objetivo |

**Criterio del análisis descriptivo:** las variables continuas se describen con las medidas
habituales (media, mediana, dispersión, forma, atípicos); las variables **dicotómicas**
se describen **únicamente mediante frecuencias y proporciones**, porque en una variable 0/1
la media es la proporción y medidas como el IQR, la asimetría o un histograma no aportan
información interpretable.

## 0. Configuración del entorno

La celda incluye dos protecciones para que el cuadernillo corra en cualquier entorno:
instala `seaborn` si falta, y restaura una función que `statsmodels ≤ 0.14.4` espera
encontrar en `scipy` pero que `scipy ≥ 1.16` ya eliminó.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Compatibilidad statsmodels <= 0.14.4 con scipy >= 1.16 -------------------
# statsmodels 0.14.4 hace `from scipy._lib._util import _lazywhere`, una función
# PRIVADA que scipy eliminó en la versión 1.16. Sin este puente, la línea
# `import statsmodels.formula.api` aborta con ImportError. Se vuelve a definir solo
# si falta; con statsmodels >= 0.15 (que ya no la usa) este bloque no hace nada.
import scipy._lib._util as _su
if not hasattr(_su, '_lazywhere'):
    def _lazywhere(cond, arrays, f, fillvalue=None, f2=None):
        """Aplica `f` donde `cond` es True y `f2` (o `fillvalue`) donde es False."""
        if (f2 is None) == (fillvalue is None):
            raise ValueError('Debe darse exactamente uno de `fillvalue` o `f2`.')
        args = np.broadcast_arrays(np.asarray(cond, dtype=bool), *arrays)
        cond, arrays = args[0], args[1:]
        temp1 = np.asarray(f(*(a[cond] for a in arrays)))
        if f2 is None:
            relleno = np.asarray(fillvalue, dtype=temp1.dtype)[()]
            out = np.full(cond.shape, relleno, dtype=temp1.dtype)
            np.place(out, cond, temp1)
        else:
            ncond = ~cond
            temp2 = np.asarray(f2(*(a[ncond] for a in arrays)))
            out = np.empty(cond.shape, dtype=np.promote_types(temp1.dtype, temp2.dtype))
            np.place(out, cond, temp1)
            np.place(out, ncond, temp2)
        return out
    _su._lazywhere = _lazywhere
# ------------------------------------------------------------------------------

try:
    import seaborn as sns
except ImportError:  # entorno sin seaborn (p. ej. Colab limpio)
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'seaborn'])
    import seaborn as sns

from scipy import stats
import statsmodels.formula.api as smf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score,
                             classification_report)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

RANDOM_STATE = 42
ALPHA = 0.05
AZUL, ROJO, VERDE, MORADO = '#4C72B0', '#C44E52', '#55A868', '#8172B3'

import scipy, statsmodels, sklearn
print('Entorno listo | pandas', pd.__version__, '| scipy', scipy.__version__,
      '| statsmodels', statsmodels.__version__, '| sklearn', sklearn.__version__)

## 1. Carga de los datos

In [ ]:
FILE_PATH = 'social_network_ads (2).csv'
df_raw = pd.read_csv(FILE_PATH)

print(f'Dimensiones: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas')
df_raw.head()

## 2. Limpieza de datos

### 2.1 Estructura y tipos de dato

In [ ]:
df_raw.info()
print('\n--- Valores únicos por columna ---')
print(df_raw.nunique())

### 2.2 Normalización de nombres y descarte del identificador

`User ID` es un identificador único (400 valores distintos en 400 filas): no contiene
información sobre el comportamiento de compra e incluirlo en el modelo solo introduciría
ruido, así que se elimina.

In [ ]:
RENOMBRES = {'User ID': 'user_id', 'Gender': 'gender', 'Age': 'age',
             'EstimatedSalary': 'estimated_salary', 'Purchased': 'purchased'}

df = df_raw.rename(columns=RENOMBRES)
df = df.drop(columns='user_id')

CONTINUAS   = ['age', 'estimated_salary']
DICOTOMICAS = ['gender', 'purchased']
TARGET      = 'purchased'

print('Columnas de trabajo:', list(df.columns))
df.head()

### 2.3 Valores faltantes y duplicados

In [ ]:
faltantes = pd.DataFrame({
    'n_nulos': df.isna().sum(),
    'pct_nulos': (df.isna().mean() * 100).round(2),
})
display(faltantes)

n_dup = df.duplicated().sum()
print(f'Filas duplicadas (tras quitar el ID): {n_dup}')
print('Nota: al eliminar el identificador, dos usuarios distintos con el mismo perfil')
print('(mismo género, edad, salario y decisión) aparecen como duplicados legítimos,')
print('por lo que NO se eliminan: son observaciones independientes.')

### 2.4 Validación de rangos y categorías

In [ ]:
chequeos = pd.DataFrame({
    'min': df[CONTINUAS].min(),
    'max': df[CONTINUAS].max(),
    'n_negativos': (df[CONTINUAS] < 0).sum(),
    'n_ceros': (df[CONTINUAS] == 0).sum(),
})
display(chequeos)

print('Edad fuera de un rango plausible [16, 100]:',
      int(((df['age'] < 16) | (df['age'] > 100)).sum()))
print('\nCategorías de gender :', df['gender'].unique().tolist())
print('Valores de purchased :', sorted(df[TARGET].unique().tolist()))

### 2.5 Detección de atípicos en las variables continuas (regla IQR)

Se marcan los puntos fuera de $[Q_1 - 1.5\,IQR,\ Q_3 + 1.5\,IQR]$. La regla se aplica
**solo a las continuas**: en una variable 0/1 carece de sentido.

In [ ]:
filas = []
for c in CONTINUAS:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (df[c] < lo) | (df[c] > hi)
    filas.append({'variable': c, 'lim_inf': lo, 'lim_sup': hi,
                  'n_outliers': int(mask.sum()), 'pct': round(mask.mean() * 100, 2)})
pd.DataFrame(filas)

### 2.6 Resumen de la limpieza

| Problema | Hallazgo | Tratamiento |
|---|---|---|
| Nombres de columna | mezcla de mayúsculas y espacios | `snake_case` |
| `User ID` | identificador sin poder predictivo | eliminado |
| Faltantes | ninguno | no se requiere imputación |
| Duplicados | perfiles repetidos | se conservan (usuarios distintos) |
| Rangos / categorías | todos válidos | sin cambios |
| Atípicos (IQR) | ninguno relevante | sin recorte |

## 3. Análisis descriptivo

Dividido según el tipo de variable, tal como se justificó en la portada.

### 3.1 Variables continuas: `Age` y `EstimatedSalary`

In [ ]:
desc = df[CONTINUAS].describe().T
desc['rango']     = desc['max'] - desc['min']
desc['IQR']       = df[CONTINUAS].quantile(0.75) - df[CONTINUAS].quantile(0.25)
desc['CV_%']      = (df[CONTINUAS].std() / df[CONTINUAS].mean() * 100).round(2)
desc['asimetria'] = df[CONTINUAS].skew()
desc['curtosis']  = df[CONTINUAS].kurt()
desc.round(3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for i, c in enumerate(CONTINUAS):
    sns.histplot(df[c], kde=True, ax=axes[i, 0], color=AZUL)
    axes[i, 0].axvline(df[c].mean(),   color='red',   ls='--', lw=1,   label='media')
    axes[i, 0].axvline(df[c].median(), color='green', ls=':',  lw=1.5, label='mediana')
    axes[i, 0].set_title(f'Distribución de {c}')
    axes[i, 0].legend(fontsize=8)

    sns.boxplot(x=df[c], ax=axes[i, 1], color=VERDE)
    axes[i, 1].set_title(f'Boxplot de {c}')
fig.tight_layout()
plt.show()

**Lectura**
- Asimetría cercana a 0 y curtosis negativa indican distribuciones bastante planas y
  simétricas, sin colas pesadas: no hace falta transformar las variables.
- El coeficiente de variación de `estimated_salary` es mucho mayor que el de `age`,
  lo que anticipa la necesidad de **estandarizar** antes de comparar coeficientes.

### 3.2 Variables dicotómicas: `Gender` y `Purchased` (proporciones)

Para estas dos variables **no** se calculan media, IQR ni histogramas: se resumen con la
**distribución de frecuencias y sus proporciones**, que es la medida descriptiva propia de
una variable binaria. Se acompaña de un **intervalo de confianza del 95 % para la
proporción** (aproximación normal), que cuantifica la precisión de cada estimación.

In [ ]:
def tabla_proporciones(serie, nombre, alpha=ALPHA):
    """Frecuencias, proporciones e IC 95% (Wald) para una variable categórica."""
    n = len(serie)
    conteo = serie.value_counts().sort_index()
    p = conteo / n
    z = stats.norm.ppf(1 - alpha / 2)
    ee = np.sqrt(p * (1 - p) / n)
    return pd.DataFrame({
        'categoria': conteo.index.astype(str),
        'frecuencia': conteo.values,
        'proporcion': p.values.round(4),
        'porcentaje': (p.values * 100).round(2),
        'IC95_inf': (p.values - z * ee.values).round(4),
        'IC95_sup': (p.values + z * ee.values).round(4),
    }).assign(variable=nombre)[['variable', 'categoria', 'frecuencia',
                                'proporcion', 'porcentaje', 'IC95_inf', 'IC95_sup']]

props = pd.concat([tabla_proporciones(df[c], c) for c in DICOTOMICAS], ignore_index=True)
display(props)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, c, col in zip(axes, DICOTOMICAS, [MORADO, ROJO]):
    p = df[c].value_counts(normalize=True).sort_index()
    ax.bar(p.index.astype(str), p.values, color=col)
    for x, v in zip(p.index.astype(str), p.values):
        ax.text(x, v + .01, f'{v:.1%}', ha='center', fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_ylabel('proporción')
    ax.set_title(f'Proporciones de {c}')
fig.tight_layout()
plt.show()

In [ ]:
# Contraste: ¿la proporción de compra difiere de 0.5? (prueba binomial exacta)
exitos = int(df[TARGET].sum())
n = len(df)
p_hat = exitos / n
try:
    p_val = stats.binomtest(exitos, n, 0.5).pvalue
except AttributeError:      # scipy < 1.7
    p_val = stats.binom_test(exitos, n, 0.5)

print(f'Proporción de compra observada : {p_hat:.4f}  ({exitos} de {n})')
print(f'Prueba binomial H0: p = 0.5     -> valor-p = {p_val:.3e}')
print('DECISIÓN:', 'se rechaza H0, las clases están desbalanceadas.'
      if p_val < ALPHA else 'no se rechaza H0.')
print(f'\nRazón de desbalance: {(1 - p_hat) / p_hat:.2f} no-compradores por cada comprador.')

> El objetivo está **desbalanceado** (~64 % / 36 %). No es un desbalance extremo, pero se
> tendrá en cuenta al elegir métricas (no basta con la *accuracy*) y al estratificar la
> partición train/test.

## 4. Análisis de relación entre variables

### 4.1 `Age` vs `EstimatedSalary` (continua vs continua)

In [ ]:
# Etiqueta legible del objetivo, solo para los gráficos (el modelo siempre usa 0/1)
ORDEN_ETIQ  = ['No compra (0)', 'Compra (1)']
PALETA_ETIQ = {'No compra (0)': AZUL, 'Compra (1)': ROJO}
df['compra_lbl'] = df[TARGET].map({0: ORDEN_ETIQ[0], 1: ORDEN_ETIQ[1]})

r, p = stats.pearsonr(df['age'], df['estimated_salary'])
print(f'Correlación de Pearson age ~ estimated_salary: r = {r:.4f} (valor-p = {p:.4f})')
fuerza = 'débil' if abs(r) < 0.3 else ('moderada' if abs(r) < 0.7 else 'fuerte')
print(f'Interpretación: relación {fuerza}', 
      f'y significativa (p < {ALPHA}).' if p < ALPHA else 'y no significativa.')
print('Una correlación débil entre predictoras es deseable: no habrá multicolinealidad.')

plt.figure(figsize=(8, 5.5))
sns.scatterplot(data=df, x='age', y='estimated_salary', hue='compra_lbl',
                hue_order=ORDEN_ETIQ, palette=PALETA_ETIQ, alpha=.8, s=45)
plt.title('Edad vs salario estimado, coloreado por compra')
plt.legend(title='')
plt.show()

> El gráfico de dispersión es clave: se aprecia que los compradores (1) se concentran en
> la zona de **mayor edad**, y dentro de los jóvenes solo compran quienes tienen salarios
> muy altos. Eso sugiere que ambas variables aportan información complementaria.

### 4.2 Variables continuas vs `Purchased` (continua vs dicotómica)

Se compara la distribución de cada continua entre quienes compraron y quienes no, con:
- **correlación punto-biserial** (equivale a Pearson cuando una variable es binaria),
- **prueba t de Welch** ($H_0$: las medias de ambos grupos son iguales),
- **Mann–Whitney U** (alternativa no paramétrica, robusta a no normalidad).

In [ ]:
filas = []
for c in CONTINUAS:
    g0 = df.loc[df[TARGET] == 0, c]
    g1 = df.loc[df[TARGET] == 1, c]
    rb, p_rb = stats.pointbiserialr(df[TARGET], df[c])
    t_stat, p_t = stats.ttest_ind(g1, g0, equal_var=False)
    u_stat, p_u = stats.mannwhitneyu(g1, g0)
    filas.append({'variable': c,
                  'media_no_compra': g0.mean(), 'media_compra': g1.mean(),
                  'diferencia': g1.mean() - g0.mean(),
                  'r_point_biserial': rb, 'p_corr': p_rb,
                  't_welch': t_stat, 'p_t': p_t,
                  'U_mannwhitney': u_stat, 'p_U': p_u})
rel = pd.DataFrame(filas)
rel['significativa'] = np.where(rel['p_U'] < ALPHA, 'Sí', 'No')
rel.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, c in zip(axes, CONTINUAS):
    sns.boxplot(data=df, x='compra_lbl', y=c, ax=ax, order=ORDEN_ETIQ,
                hue='compra_lbl', hue_order=ORDEN_ETIQ, palette=PALETA_ETIQ, legend=False)
    sns.stripplot(data=df, x='compra_lbl', y=c, ax=ax, order=ORDEN_ETIQ,
                  color='k', alpha=.25, size=3)
    ax.set_title(f'{c} según la decisión de compra')
    ax.set_xlabel('')
fig.tight_layout()
plt.show()

### 4.3 `Gender` vs `Purchased` (dicotómica vs dicotómica)

Al ser ambas categóricas, la relación se estudia con una **tabla de contingencia de
proporciones condicionales** y la prueba **chi-cuadrado de independencia**:

- $H_0$: el género y la decisión de compra son independientes.
- $H_1$: existe asociación entre ambas variables.

In [ ]:
tabla = pd.crosstab(df['gender'], df[TARGET])
print('--- Frecuencias observadas ---')
display(tabla)

print('--- Proporción de compra DENTRO de cada género (condicional por fila) ---')
display((pd.crosstab(df['gender'], df[TARGET], normalize='index') * 100).round(2))

chi2, p_chi, gl, esperadas = stats.chi2_contingency(tabla)
n = tabla.values.sum()
v_cramer = np.sqrt(chi2 / (n * (min(tabla.shape) - 1)))
print(f'\nChi-cuadrado = {chi2:.4f}   gl = {gl}   valor-p = {p_chi:.4f}')
print(f'V de Cramér  = {v_cramer:.4f}  (0 = sin asociación, 1 = asociación total)')
print('DECISIÓN:', 'se rechaza H0: hay asociación entre género y compra.' if p_chi < ALPHA
      else 'no se rechaza H0: no hay evidencia de asociación entre género y compra.')

In [ ]:
plt.figure(figsize=(7, 4.5))
prop_g = pd.crosstab(df['gender'], df[TARGET], normalize='index')[1]
plt.bar(prop_g.index, prop_g.values, color=[MORADO, VERDE])
for x, v in zip(prop_g.index, prop_g.values):
    plt.text(x, v + .01, f'{v:.1%}', ha='center')
plt.axhline(df[TARGET].mean(), color='k', ls='--', lw=1, label='proporción global')
plt.ylabel('proporción que compra')
plt.title('Proporción de compra por género')
plt.legend()
plt.show()

### 4.4 Matriz de correlación general

Se codifica `gender` como 0/1 para poder incluirlo. Con variables binarias, la correlación
de Pearson equivale a la punto-biserial (continua–binaria) o al coeficiente phi
(binaria–binaria), de modo que la matriz es una **síntesis visual** de las secciones 4.1–4.3.

In [ ]:
df_corr = df.drop(columns=['gender', 'compra_lbl'])
df_corr['gender_male'] = (df['gender'] == 'Male').astype(int)

corr = df_corr.corr()
plt.figure(figsize=(6.5, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, square=True, cbar_kws={'shrink': .8})
plt.title('Matriz de correlación')
plt.show()

print('Correlación de cada predictora con la compra:')
print(corr[TARGET].drop(TARGET).sort_values(ascending=False).round(4).to_string())

## 5. Preparación de los datos para el modelo

### 5.1 Codificación y partición estratificada

La partición se hace **estratificada por `purchased`** para que train y test conserven la
misma proporción de compradores, algo importante dado el desbalance detectado en 3.2.

In [ ]:
modelo_df = df.copy()
modelo_df['gender_male'] = (modelo_df['gender'] == 'Male').astype(int)   # Female = 0
PREDICTORAS = ['age', 'estimated_salary', 'gender_male']

X = modelo_df[PREDICTORAS]
y = modelo_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

resumen = pd.DataFrame({
    'n': [len(y_train), len(y_test)],
    'compradores': [int(y_train.sum()), int(y_test.sum())],
    'prop_compra': [y_train.mean(), y_test.mean()],
}, index=['train', 'test'])
display(resumen.round(4))

### 5.2 Estandarización

`age` (18–60 años) y `estimated_salary` (15 000–150 000) viven en escalas muy distintas.
Se crea una versión estandarizada ($z = (x-\bar{x})/s$) **ajustando el escalador solo con
`train`** para no filtrar información del test. Se usará en la sección 6.3 para comparar la
importancia relativa de las variables; el modelo principal se ajusta en las unidades
originales porque así los coeficientes son directamente interpretables.

In [ ]:
scaler = StandardScaler().fit(X_train[CONTINUAS])

X_train_z = X_train.copy()
X_test_z  = X_test.copy()
X_train_z[CONTINUAS] = scaler.transform(X_train[CONTINUAS])
X_test_z[CONTINUAS]  = scaler.transform(X_test[CONTINUAS])

print('Medias y desviaciones aprendidas del train:')
print(pd.DataFrame({'media': scaler.mean_, 'desv_est': scaler.scale_}, index=CONTINUAS).round(2))
X_train_z.head()

## 6. Modelo de regresión logística

El modelo estima la probabilidad de compra mediante la función logística:

$$P(Y=1 \mid X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X_1 + \dots + \beta_k X_k)}}$$

equivalentemente, en escala de *log-odds* (logit), el modelo es lineal:

$$\ln\!\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1\,\text{age} + \beta_2\,\text{salary} + \beta_3\,\text{gender\_male}$$

Los coeficientes se estiman por **máxima verosimilitud** (no por mínimos cuadrados).

### 6.1 Ajuste e inferencia (statsmodels)

In [ ]:
train_sm = X_train.copy()
train_sm[TARGET] = y_train.values

formula = f'{TARGET} ~ age + estimated_salary + gender_male'
logit = smf.logit(formula, data=train_sm).fit()
print(logit.summary())

### 6.2 Coeficientes, *odds ratios* e interpretación

El coeficiente $\beta_j$ está en *log-odds*; su exponencial $e^{\beta_j}$ es el **odds ratio**:
el factor por el que se multiplica la razón de probabilidades al aumentar una unidad esa
variable, manteniendo las demás constantes.

- $OR > 1$ → la variable **aumenta** la probabilidad de compra.
- $OR < 1$ → la **reduce**.
- $OR = 1$ (o IC que contiene a 1) → sin efecto significativo.

In [ ]:
ic = logit.conf_int()
ic.columns = ['ic_inf', 'ic_sup']
coef = pd.DataFrame({
    'coef_log_odds': logit.params,
    'error_est': logit.bse,
    'z': logit.tvalues,
    'valor_p': logit.pvalues,
    'odds_ratio': np.exp(logit.params),
    'OR_IC95_inf': np.exp(ic['ic_inf']),
    'OR_IC95_sup': np.exp(ic['ic_sup']),
})
coef['significativa'] = np.where(coef['valor_p'] < ALPHA, 'Sí', 'No')
display(coef.round(6))

print('Efecto de un cambio de escala práctica:')
print(f"  +10 años de edad      -> OR = {np.exp(logit.params['age'] * 10):.3f}")
print(f"  +$10,000 de salario   -> OR = {np.exp(logit.params['estimated_salary'] * 10000):.3f}")

In [ ]:
# Lectura en palabras de los coeficientes
unidades = {'age': ('1 año más de edad', 1),
            'estimated_salary': ('$10,000 más de salario', 10000),
            'gender_male': ('ser hombre en lugar de mujer', 1)}

print('INTERPRETACIÓN DE LOS COEFICIENTES\n' + '=' * 70)
for v, (texto, esc) in unidades.items():
    b = logit.params[v]
    orr = np.exp(b * esc)
    p = logit.pvalues[v]
    if p < ALPHA:
        signo = 'AUMENTA' if orr > 1 else 'DISMINUYE'
        cambio = (orr - 1) * 100
        print(f'- {texto}: OR = {orr:.3f} (p = {p:.4f}) -> {signo} la razón de',
              f'probabilidades de compra en {abs(cambio):.1f} %, ceteris paribus.')
    else:
        print(f'- {texto}: OR = {orr:.3f} (p = {p:.4f}) -> NO significativo al {ALPHA:.0%};',
              'no hay evidencia de que modifique la probabilidad de compra.')

### 6.3 Importancia relativa: coeficientes estandarizados

Como `age` y `estimated_salary` tienen escalas distintas, sus coeficientes crudos no son
comparables entre sí. Al reajustar el modelo sobre las variables **estandarizadas**, cada
coeficiente mide el efecto de **una desviación estándar**, lo que sí permite compararlos.

In [ ]:
train_z_sm = X_train_z.copy()
train_z_sm[TARGET] = y_train.values
logit_z = smf.logit(formula, data=train_z_sm).fit(disp=0)

comp = pd.DataFrame({
    'coef_estandarizado': logit_z.params,
    'OR_por_1_desv_est': np.exp(logit_z.params),
    'valor_p': logit_z.pvalues,
}).drop(index='Intercept')
comp['importancia_abs'] = comp['coef_estandarizado'].abs()
display(comp.sort_values('importancia_abs', ascending=False).round(4))

orden = comp.sort_values('coef_estandarizado')
plt.figure(figsize=(8, 3.5))
plt.barh(orden.index, orden['coef_estandarizado'],
         color=[ROJO if v < 0 else AZUL for v in orden['coef_estandarizado']])
plt.axvline(0, color='k', lw=.8)
plt.title('Coeficientes estandarizados (log-odds por desviación estándar)')
plt.show()

### 6.4 Significancia global del modelo

Prueba de **razón de verosimilitud** (LR test):
- $H_0$: $\beta_1 = \beta_2 = \beta_3 = 0$ (el modelo no explica nada mejor que el modelo nulo)
- $H_1$: al menos un coeficiente es distinto de cero

In [ ]:
lr_stat = logit.llr
lr_p    = logit.llr_pvalue
gl      = int(logit.df_model)
chi_crit = stats.chi2.ppf(1 - ALPHA, gl)

print(f'Log-verosimilitud modelo nulo   : {logit.llnull:.4f}')
print(f'Log-verosimilitud modelo ajustado: {logit.llf:.4f}')
print(f'Estadístico LR = {lr_stat:.4f}   gl = {gl}   chi2 crítico = {chi_crit:.4f}')
print(f'valor-p = {lr_p:.4e}')
print('DECISIÓN:', 'se RECHAZA H0: el modelo es globalmente significativo.'
      if lr_p < ALPHA else 'NO se rechaza H0.')

print(f'\nPseudo R2 de McFadden = {logit.prsquared:.4f}')
print('(valores entre 0.2 y 0.4 ya indican un ajuste excelente en este indicador)')

### 6.5 Versión `scikit-learn` y verificación de equivalencia

Se ajusta el mismo modelo con `scikit-learn` sobre los datos estandarizados (prácticamente
sin regularización, `C` muy grande) y se comprueba que produce **las mismas probabilidades**
que `statsmodels`. Confirmado esto, las métricas de la sección 7 son válidas para ambos.

In [ ]:
sk_logit = LogisticRegression(C=1e6, solver='lbfgs', max_iter=1000)
sk_logit.fit(X_train_z, y_train)

p_sm = logit.predict(X_test)                       # statsmodels, escala original
p_sk = sk_logit.predict_proba(X_test_z)[:, 1]      # sklearn, escala estandarizada

print(f'Máxima diferencia absoluta entre probabilidades: {np.abs(p_sm - p_sk).max():.8f}')
print(f'Correlación entre ambas predicciones          : {np.corrcoef(p_sm, p_sk)[0, 1]:.6f}')
print('\nSon el mismo modelo: la estandarización cambia la escala de los coeficientes,')
print('no las probabilidades predichas.')

## 7. Métricas de evaluación

Todas se calculan sobre el conjunto **test**, que no intervino en el ajuste.

### 7.1 Matriz de confusión

In [ ]:
UMBRAL = 0.5
y_prob = p_sm
y_pred = (y_prob >= UMBRAL).astype(int)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Pred. 0', 'Pred. 1'], yticklabels=['Real 0', 'Real 1'])
plt.title(f'Matriz de confusión (umbral = {UMBRAL})')
plt.show()

print(f'Verdaderos negativos (TN) = {tn}  -> no compró y se predijo que no')
print(f'Falsos positivos     (FP) = {fp}  -> no compró pero se predijo que sí')
print(f'Falsos negativos     (FN) = {fn}  -> compró pero se predijo que no')
print(f'Verdaderos positivos (TP) = {tp}  -> compró y se predijo que sí')

### 7.2 Métricas derivadas

| Métrica | Fórmula | Qué responde |
|---|---|---|
| Exactitud (*accuracy*) | $(TP+TN)/n$ | ¿qué proporción del total acierta? |
| Precisión | $TP/(TP+FP)$ | de los que predije compradores, ¿cuántos lo eran? |
| Sensibilidad (*recall*) | $TP/(TP+FN)$ | de los compradores reales, ¿cuántos detecté? |
| Especificidad | $TN/(TN+FP)$ | de los no compradores, ¿cuántos identifiqué bien? |
| F1-score | media armónica de precisión y recall | equilibrio entre ambas |

In [ ]:
especificidad = tn / (tn + fp)
metricas = pd.DataFrame({
    'valor': [accuracy_score(y_test, y_pred),
              precision_score(y_test, y_pred),
              recall_score(y_test, y_pred),
              especificidad,
              f1_score(y_test, y_pred),
              roc_auc_score(y_test, y_prob),
              average_precision_score(y_test, y_prob)]},
    index=['Exactitud (accuracy)', 'Precisión', 'Sensibilidad (recall)',
           'Especificidad', 'F1-score', 'AUC-ROC', 'AUC-PR (average precision)'])
display(metricas.round(4))

print('Línea base (predecir siempre la clase mayoritaria):',
      f'{max(y_test.mean(), 1 - y_test.mean()):.4f} de exactitud\n')
print(classification_report(y_test, y_pred, target_names=['No compra (0)', 'Compra (1)']))

### 7.3 Curva ROC y curva Precision-Recall

La **ROC** muestra el compromiso entre sensibilidad y (1 − especificidad) para todos los
umbrales; el **AUC** es la probabilidad de que el modelo asigne mayor puntaje a un comprador
elegido al azar que a un no comprador. La curva **Precision-Recall** es más informativa
cuando las clases están desbalanceadas, porque se centra en la clase positiva.

In [ ]:
fpr, tpr, thr_roc = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
prec, rec, thr_pr = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(fpr, tpr, color=AZUL, lw=2, label=f'Regresión logística (AUC = {auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Clasificador aleatorio (AUC = 0.5)')
axes[0].set_xlabel('1 - Especificidad (FPR)'); axes[0].set_ylabel('Sensibilidad (TPR)')
axes[0].set_title('Curva ROC'); axes[0].legend(loc='lower right')

axes[1].plot(rec, prec, color=ROJO, lw=2, label=f'AP = {ap:.4f}')
axes[1].axhline(y_test.mean(), color='k', ls='--', lw=1,
                label=f'Línea base = {y_test.mean():.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precisión')
axes[1].set_title('Curva Precision-Recall'); axes[1].legend(loc='lower left')
fig.tight_layout()
plt.show()

### 7.4 Determinación del punto de corte (*cutoff*)

El 0.5 es una **convención**, no un resultado del modelo: la regresión logística devuelve una
probabilidad continua, y el punto de corte es una decisión aparte que fija cómo se traduce
esa probabilidad en un 0 o un 1. Moverlo cambia el equilibrio entre los dos tipos de error:

- **Bajar** el cutoff → más predicciones de compra → sube la sensibilidad, bajan la
  especificidad y la precisión (más falsos positivos).
- **Subir** el cutoff → menos predicciones de compra → lo contrario (más falsos negativos).

Se evalúan tres criterios sobre una rejilla fina de cortes:

| Criterio | Qué maximiza | Cuándo conviene |
|---|---|---|
| **Índice J de Youden** | $\text{sens} + \text{esp} - 1$ | tratar ambos errores como igual de graves |
| **F1 máximo** | equilibrio precisión–recall | interesa sobre todo acertar la clase positiva |
| **Cruce sens = esp** | igualar ambas tasas | se quiere el mismo acierto en los dos grupos |

In [ ]:
cortes = np.linspace(0.01, 0.99, 197)
filas = []
for t in cortes:
    yp = (y_prob >= t).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_test, yp, labels=[0, 1]).ravel()
    sens = tp_ / (tp_ + fn_) if (tp_ + fn_) else 0.0
    esp  = tn_ / (tn_ + fp_) if (tn_ + fp_) else 0.0
    filas.append({'cutoff': t, 'sensibilidad': sens, 'especificidad': esp,
                  'J_youden': sens + esp - 1,
                  'exactitud': accuracy_score(y_test, yp),
                  'precision': precision_score(y_test, yp, zero_division=0),
                  'F1': f1_score(y_test, yp, zero_division=0),
                  'FP': fp_, 'FN': fn_})
curva = pd.DataFrame(filas)
curva['dif_sens_esp'] = (curva['sensibilidad'] - curva['especificidad']).abs()

c_youden = curva.loc[curva['J_youden'].idxmax(), 'cutoff']
c_f1     = curva.loc[curva['F1'].idxmax(), 'cutoff']
c_cruce  = curva.loc[curva['dif_sens_esp'].idxmin(), 'cutoff']

print(f'Cutoff por índice de Youden : {c_youden:.3f}  (J = {curva["J_youden"].max():.4f})')
print(f'Cutoff por F1 máximo        : {c_f1:.3f}  (F1 = {curva["F1"].max():.4f})')
print(f'Cutoff donde sens = esp     : {c_cruce:.3f}')
display(curva[curva['cutoff'].round(2).isin([0.20, 0.30, 0.40, 0.46, 0.50, 0.60, 0.70, 0.80])]
        .drop(columns='dif_sens_esp').round(4).to_string(index=False))

#### Gráfico del *cutoff*

Tres vistas del mismo problema: dónde se cruzan sensibilidad y especificidad, dónde se
maximizan los criterios de decisión, y dónde queda ese punto sobre la curva ROC.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# (a) sensibilidad vs especificidad: el cruce iguala el acierto en ambas clases
ax = axes[0]
ax.plot(curva['cutoff'], curva['sensibilidad'], color=ROJO, lw=2,
        label='Sensibilidad (detecta compradores)')
ax.plot(curva['cutoff'], curva['especificidad'], color=AZUL, lw=2,
        label='Especificidad (detecta no compradores)')
ax.axvline(c_cruce, color='k', ls='--', lw=1.2)
ax.annotate(f'cruce = {c_cruce:.2f}', xy=(c_cruce, 0.5),
            xytext=(c_cruce + 0.06, 0.32), fontsize=9,
            arrowprops=dict(arrowstyle='->', lw=1))
ax.set_xlabel('Punto de corte'); ax.set_ylabel('Tasa')
ax.set_title('(a) Sensibilidad vs especificidad')
ax.legend(fontsize=8, loc='center left')

# (b) criterios de decisión
ax = axes[1]
ax.plot(curva['cutoff'], curva['exactitud'], color=VERDE, lw=2, label='Exactitud')
ax.plot(curva['cutoff'], curva['F1'], color=MORADO, lw=2, label='F1')
ax.plot(curva['cutoff'], curva['J_youden'], color='#D1913C', lw=2, label='J de Youden')
ax.axvline(c_youden, color='k', ls='--', lw=1.2, label=f'óptimo = {c_youden:.2f}')
ax.axvline(0.5, color='gray', ls=':', lw=1.2, label='convencional = 0.50')
ax.set_xlabel('Punto de corte'); ax.set_ylabel('Valor de la métrica')
ax.set_title('(b) Criterios de decisión')
ax.legend(fontsize=8, loc='lower center')

# (c) el cutoff elegido sobre la ROC
ax = axes[2]
ax.plot(fpr, tpr, color=AZUL, lw=2, label=f'ROC (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
i_y = int(np.argmax(tpr - fpr))
ax.scatter(fpr[i_y], tpr[i_y], s=130, color=ROJO, zorder=5, edgecolor='w',
           label=f'cutoff óptimo = {thr_roc[i_y]:.2f}')
ax.plot([fpr[i_y], fpr[i_y]], [fpr[i_y], tpr[i_y]], color=ROJO, lw=2, ls=':')
ax.annotate(f'J = {(tpr - fpr).max():.3f}',
            xy=(fpr[i_y], (tpr[i_y] + fpr[i_y]) / 2),
            xytext=(fpr[i_y] + 0.13, (tpr[i_y] + fpr[i_y]) / 2 - 0.08), fontsize=9,
            arrowprops=dict(arrowstyle='->', lw=1))
ax.set_xlabel('1 - Especificidad'); ax.set_ylabel('Sensibilidad')
ax.set_title('(c) Punto de corte sobre la ROC')
ax.legend(fontsize=8, loc='lower right')

fig.tight_layout()
plt.show()

#### Elección final y comparación con el umbral 0.5

In [ ]:
CUTOFF = round(float(c_youden), 2)

def evaluar(t, nombre):
    yp = (y_prob >= t).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_test, yp, labels=[0, 1]).ravel()
    return {'escenario': nombre, 'cutoff': t,
            'exactitud': accuracy_score(y_test, yp),
            'precision': precision_score(y_test, yp, zero_division=0),
            'sensibilidad': recall_score(y_test, yp),
            'especificidad': tn_ / (tn_ + fp_),
            'F1': f1_score(y_test, yp),
            'FP (falsas alarmas)': fp_, 'FN (compradores perdidos)': fn_}

comparacion = pd.DataFrame([evaluar(0.5, 'Convencional'),
                            evaluar(CUTOFF, 'Óptimo (Youden)')])
display(comparacion.set_index('escenario').round(4))

num = comparacion.drop(columns='escenario')
delta = num.iloc[1] - num.iloc[0]
print(f'Al mover el corte de 0.50 a {CUTOFF}:')
print(f'  - se recuperan {int(-delta["FN (compradores perdidos)"])} compradores'
      f' que antes se perdían,')
print(f'  - a cambio de {int(delta["FP (falsas alarmas)"])} falsas alarmas adicionales,')
print(f'  - la exactitud pasa de {comparacion.iloc[0]["exactitud"]:.4f}'
      f' a {comparacion.iloc[1]["exactitud"]:.4f}.')
print(f'\nCUTOFF ELEGIDO = {CUTOFF}')

> **Decisión:** se adopta el cutoff óptimo por Youden. En este caso los tres criterios
> (J de Youden, F1 y exactitud) apuntan al mismo valor, lo que da confianza en la elección.
> El cruce sensibilidad = especificidad cae más abajo: sería el corte a usar si la prioridad
> fuese detectar la mayor cantidad posible de compradores, aceptando más falsas alarmas.
>
> Aun así, la diferencia frente a 0.5 es pequeña, señal de que el modelo separa bien las
> clases y no depende de un ajuste fino del corte.

### 7.5 Visualización de la frontera de decisión

Con solo dos predictoras continuas se puede dibujar la región donde el modelo predice
compra. La frontera es una **recta en el espacio (edad, salario)**, porque el modelo es
lineal en el logit: ahí es donde la probabilidad estimada vale exactamente 0.5.

In [ ]:
gm = int(round(X_train['gender_male'].mean()))   # género más frecuente en train
a_min, a_max = df['age'].min() - 2, df['age'].max() + 2
s_min, s_max = df['estimated_salary'].min() - 5000, df['estimated_salary'].max() + 5000
aa, ss = np.meshgrid(np.linspace(a_min, a_max, 300), np.linspace(s_min, s_max, 300))
grid = pd.DataFrame({'age': aa.ravel(), 'estimated_salary': ss.ravel(),
                     'gender_male': gm})
zz = logit.predict(grid).values.reshape(aa.shape)

plt.figure(figsize=(9, 6))
plt.contourf(aa, ss, zz, levels=np.linspace(0, 1, 21), cmap='RdBu_r', alpha=.55)
plt.colorbar(label='Probabilidad estimada de compra')
cs1 = plt.contour(aa, ss, zz, levels=[0.5], colors='k', linewidths=2)
cs2 = plt.contour(aa, ss, zz, levels=[CUTOFF], colors='#D1913C',
                  linewidths=2, linestyles='--')
plt.clabel(cs1, fmt={0.5: 'corte 0.50'}, fontsize=8)
plt.clabel(cs2, fmt={CUTOFF: f'corte {CUTOFF}'}, fontsize=8)
sns.scatterplot(data=df, x='age', y='estimated_salary', hue='compra_lbl',
                hue_order=ORDEN_ETIQ,
                palette={ORDEN_ETIQ[0]: '#1f4e79', ORDEN_ETIQ[1]: '#8b0000'},
                edgecolor='w', s=45)
plt.title('Frontera de decisión (línea negra = probabilidad 0.5)')
plt.legend(title='', loc='upper left')
plt.show()

## 8. Interpretación y conclusiones

In [ ]:
print('=' * 72)
print('RESUMEN AUTOMÁTICO DEL ANÁLISIS'.center(72))
print('=' * 72)
print(f'\n1. DATOS')
print(f'   - {len(df)} usuarios, sin faltantes ni valores inválidos.')
print(f'   - Proporción de compra: {df[TARGET].mean():.1%} '
      f'(clases desbalanceadas ~{(1-df[TARGET].mean())/df[TARGET].mean():.1f} a 1).')
print(f'   - Proporción de hombres: {(df["gender"] == "Male").mean():.1%}')

print(f'\n2. RELACIONES DETECTADAS EN EL EDA')
for c in CONTINUAS:
    rb, p_rb = stats.pointbiserialr(df[TARGET], df[c])
    print(f'   - {c:18s}: r punto-biserial = {rb:+.3f} (p = {p_rb:.2e})')
print(f'   - gender vs purchased: chi2 p = {p_chi:.4f} -> '
      f'{"asociación significativa" if p_chi < ALPHA else "sin asociación significativa"}')

print(f'\n3. MODELO')
print(f'   - Globalmente significativo: LR p = {lr_p:.2e}')
print(f'   - Pseudo R2 McFadden = {logit.prsquared:.4f}')
sig = coef[(coef['valor_p'] < ALPHA)].drop(index='Intercept', errors='ignore')
print(f'   - Predictoras significativas: {", ".join(sig.index) if len(sig) else "ninguna"}')

print(f'\n4. DESEMPEÑO EN TEST (umbral {UMBRAL})')
for nombre, valor in metricas['valor'].items():
    print(f'   - {nombre:28s}: {valor:.4f}')

print(f'\n5. PUNTO DE CORTE')
print(f'   - Convencional: 0.50 | Óptimo por Youden: {CUTOFF}')
print(f'   - Exactitud con el corte óptimo: '
      f'{comparacion.iloc[1]["exactitud"]:.4f}')
print('=' * 72)

### 8.1 Conclusiones

**Sobre los datos y el EDA**
- El dataset está completo y limpio: sin faltantes, sin valores fuera de rango y sin
  atípicos que ameriten tratamiento.
- La variable objetivo está desbalanceada (≈ 36 % de compradores), por lo que la exactitud
  por sí sola es insuficiente: se reportan también recall, F1 y AUC.
- `age` es la variable más asociada a la compra, seguida de `estimated_salary`.
  El género **no** muestra asociación significativa con la decisión de compra
  (chi-cuadrado no significativo), y el gráfico de proporciones lo confirma: hombres y
  mujeres compran en proporciones muy similares.
- `age` y `estimated_salary` muestran una correlación **débil** entre sí (r ≈ 0.16):
  aunque es estadísticamente significativa por el tamaño muestral, su magnitud es tan baja
  que aportan información complementaria y no generan multicolinealidad.

**Sobre el modelo**
- El modelo es globalmente significativo y el pseudo-R² de McFadden indica un buen ajuste.
- Los *odds ratios* muestran que tanto la edad como el salario **aumentan** la probabilidad
  de compra, mientras que `gender_male` no resulta significativo — coherente con el EDA.
- En coeficientes estandarizados, la **edad** es el predictor dominante.

**Sobre las métricas**
- El AUC-ROC muestra una capacidad discriminativa alta: el modelo ordena correctamente a
  compradores frente a no compradores en la gran mayoría de los pares.
- La exactitud supera claramente a la línea base de predecir siempre "no compra".
- El recall indica qué fracción de compradores reales se captaría en una campaña dirigida;
  si el objetivo del negocio fuera no perder clientes potenciales, bajar el umbral por
  debajo de 0.5 aumentaría el recall a costa de la precisión.

**Limitaciones**
- Solo 400 observaciones y 3 predictoras: el modelo es simple y el test tiene 80 casos,
  por lo que las métricas tienen una incertidumbre no despreciable.
- La frontera de decisión es lineal en el logit; el gráfico de dispersión sugiere que una
  frontera curva (o términos de interacción `age × salary`) podría capturar mejor el
  patrón de los jóvenes con salarios altos.
- `gender` podría mantenerse fuera del modelo final por no aportar información.